# Masked-Diffusion BabyLM — Strict-Small (Training)

Train a LLaDA/MDLM-style masked-diffusion language model on the BabyLM 2026
**Strict-Small** corpus (≤10M words, ≤10 epochs, ≤100M words seen).

Everything heavy (data, tokenizer, runs) lives on Google Drive so a Colab
disconnect never loses work. Data prep is skipped if already done, and training
**auto-resumes** from the last checkpoint — re-run the training cell after any
disconnect (even on another day) and it continues where it left off. Run the
cells top to bottom.

**Before you start:** add an `HF_TOKEN` (read access) via the Colab Secrets panel (key icon).

In [ ]:
# Step 1 — Parameters (edit these)
REPO_URL   = "https://github.com/Amos-Luna/Masked-Diffusion-BabyLM.git"
DRIVE_ROOT = "/content/drive/MyDrive/Researchs/BabyLM_diffusion_G4"
CONDITION  = "MD_base"   # MD_base | MD_freq_mask | MD_layerdup
SEED       = 42

In [ ]:
# Step 2 — Mount Drive + clone/pull the repo
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
if not os.path.isdir("/content/Masked-Diffusion-BabyLM"):
    subprocess.run(["git", "clone", REPO_URL, "/content/Masked-Diffusion-BabyLM"], check=True)
%cd /content/Masked-Diffusion-BabyLM/diffusion
subprocess.run(["git", "pull"], check=False)

In [ ]:
# Step 3 — HuggingFace token (from Colab Secrets)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets (key icon).", e)

In [ ]:
# Step 4 — Bootstrap: install + Drive symlinks + data + smoke test
!bash scripts/colab_bootstrap.sh --drive-root "$DRIVE_ROOT"

In [ ]:
# Step 5 — Train. Checkpoints (model + optimizer state) are written to
# runs/<run>/checkpoints/ on Drive. If Colab disconnects, just RE-RUN this cell:
# it auto-detects the latest checkpoint for this (condition, seed) and CONTINUES
# from there — even on a different day. Use --no-resume to force a fresh run.
!python scripts/train.py --condition $CONDITION --seed $SEED -v \
    --token-data data/tokens --tokenizer tokenizer/mdlm_bpe_16k

In [ ]:
# Step 6 — Training curves from the run log. The weighted MDLM loss is very
# noisy (it is reweighted by 1/t), so the UNWEIGHTED cross-entropy is the curve
# to watch for real progress. We also plot the LR schedule (warmup + cosine).
# The PNG is SAVED into the run dir on Drive so it persists for the paper.
import glob, json, os
import matplotlib.pyplot as plt

# Pick the latest REAL run (skip the bootstrap smoke run, which also matches).
runs = [d for d in sorted(glob.glob(f"runs/*_{CONDITION}_seed{SEED}"))
        if not os.path.basename(d).startswith("_smoke")]
run = runs[-1]
steps, train, ce, lr = [], [], [], []
eval_steps, val, val_ce = [], [], []
for line in open(f"{run}/log.jsonl"):
    r = json.loads(line)
    if r["phase"] == "train":
        steps.append(r["step"]); train.append(r["avg_train_loss"])
        ce.append(r.get("avg_ce_unweighted")); lr.append(r.get("lr"))
    else:
        eval_steps.append(r["step"]); val.append(r["val_loss"])
        val_ce.append(r.get("val_ce_unweighted"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(steps, train, alpha=0.35, label="train loss (weighted 1/t)")
if any(c is not None for c in ce):
    ax1.plot(steps, ce, label="train CE (unweighted)")
if any(c is not None for c in val_ce):
    ax1.plot(eval_steps, val_ce, "o-", label="val CE (unweighted)")
else:
    ax1.plot(eval_steps, val, "o-", label="val loss (weighted)")
ax1.set_xlabel("step"); ax1.set_ylabel("loss / CE"); ax1.legend(); ax1.set_title(os.path.basename(run))
if any(v is not None for v in lr):
    ax2.plot(steps, lr); ax2.set_xlabel("step"); ax2.set_ylabel("learning rate")
    ax2.set_title("LR schedule (warmup + cosine)")

os.makedirs(f"{run}/figures", exist_ok=True)
out = f"{run}/figures/train_curve.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
print("Saved", out)   # lives on Drive — survives disconnects
plt.show()